In [2]:
# Imports
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

ModuleNotFoundError: No module named 'numpy'

In [ ]:
#Verifying GPU Availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
#CNN's need tensor inputs, not as PILLOW images, thus they need to be converted using ToTensor.
#The tensors are then changed from [0,255] to [0,1] to make it easier to normalize.
transform = transforms.Compose([
    transforms.ToTensor(),
])

#Fashion MNIST Training Data
Fashion_mnist_train_validation = datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)
Fashion_mnist_train , Fashion_mnist_validate  = random_split(
    Fashion_mnist_train_validation, [50000, 10000], generator=torch.Generator().manual_seed(1)
)

#Fashion MNIST Testing Data
Fashion_mnist_test = datasets.FashionMNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

#DataLoader divides the dataset into mini-batches.
#shuffle randomizes the chosen sets.
Fashion_mnist_train_loader = DataLoader(
    Fashion_mnist_train,
    batch_size=128,
    shuffle=True
)
Fashion_mnist_validate_loader = DataLoader(
    Fashion_mnist_validate,
    batch_size=128,
    shuffle=False
)
Fashion_mnist_test_loader = DataLoader(
    Fashion_mnist_test,
    batch_size=128,
    shuffle=False
)

In [ ]:
#Defining the Classes
class_names =["T-shirt/top","Trouser", "Pullover","Dress","Coat", "Sandal","Shirt","Sneaker","Bag", "Ankle boot"]

#Verification of dataset


for i in range(5):
  image,label=Fashion_mnist_train[i]
  plt.figure()
  # Squeeze or rearrange dimensions because PyTorch uses [Channels, Height, Width]
  plt.imshow(image.squeeze(), cmap='gray')
  plt.title(f"Label: {class_names[label]}")
  plt.axis('off')
  plt.show()

In [ ]:
#Model
class MyModel(nn.Module):
    def __init__(self):
        super(MyModel,self).__init__()
        # Input Layer as per given model
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(784, 16)
        self.left_fc1 = nn.Linear(16, 8)
        self.left_fc2 = nn.Linear(8, 8)
        self.right_fc1 = nn.Linear(16, 12)
        self.right_fc2 = nn.Linear(12, 8)
        self.output = nn.Linear(16, 10)

    def forward(self,x):
        x=self.flatten(x)
        x=F.relu(self.fc1(x))
        left=F.relu(self.left_fc1(x))
        skip=left
        left=F.relu(self.left_fc2(left))
        left=left+skip
        right=F.relu(self.right_fc1(x))
        right=F.relu(self.right_fc2(right))
        merged=torch.cat((left,right),dim=1)
        output=self.output(merged)
        return output

In [ ]:
model = MyModel().to(device)

#Standard loss function
loss = nn.CrossEntropyLoss()

#Adam automatically changes the learning rate dependent on whats best
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
train_losses = []
epochs = 50
for epoch in range(epochs):
    model.train()
    running_loss = 0

    for images, labels in Fashion_mnist_train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        losses = loss(outputs, labels)

        losses.backward()

        optimizer.step()

        running_loss += losses.item()

    epoch_loss = running_loss / len(Fashion_mnist_train_loader)

    train_losses.append(epoch_loss)
    model.eval()
    val_loss=0
    total=0
    correct=0

    with torch.no_grad():
        for images, labels in Fashion_mnist_validate_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            current_loss = loss(outputs, labels)
            val_loss+=current_loss.item()*images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    epoch_val_loss = val_loss / len(Fashion_mnist_validate_loader.dataset)
    val_accuracy = 100 * correct / total

    print(f"Epoch [{epoch+1}/{epochs}] -> "
          f"Train Loss: {epoch_loss:.4f} | "
          f"Val Loss: {epoch_val_loss:.4f} | "
          f"Val Acc: {val_accuracy:.2f}%")

In [ ]:
#Training LOSS CURVE
plt.figure(figsize=(4,3))
plt.title("Training Loss Curve")
plt.plot(train_losses)
plt.xlabel("Epochs")
plt.ylabel("Losses")
plt.grid()
plt.show()

#Validation LOSS CURVE
plt.figure(figsize=(4,3))
plt.title("Validation Loss Curve")
plt.plot( epoch_val_loss)
plt.xlabel("Epochs")
plt.ylabel("Losses")
plt.grid()
plt.show()

#Validation Accuracy CURVE
plt.figure(figsize=(4,3))
plt.title("Validation Accuacy Curve")
plt.plot( val_accuracy)
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.grid()
plt.show()


In [ ]:
def evaluate(model, loader):

    model.eval()
    y_true = []
    y_pred = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            y_true.extend(labels.numpy())
            y_pred.extend(preds.cpu().numpy())

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true,y_pred,average="macro")
    rec = recall_score(y_true,y_pred,average="macro")
    f1 = f1_score(y_true,y_pred,average="macro")w
    return acc, prec, rec, f1, y_true, y_pred

In [ ]:
#Fashion_MNIST TESTING
Fashion_mnist_acc, Fashion_mnist_prec, Fashion_mnist_rec, Fashion_mnist_f1,y_true, y_pred = evaluate(
    model,
    Fashion_mnist_test_loader
)

print("FASHION MNIST TEST")
print(f"Accuracy : {Fashion_mnist_acc:.6f}")
print(f"Precision: {Fashion_mnist_prec:.6f}")
print(f"Recall   : {Fashion_mnist_rec:.6f}")
print(f"F1       : {Fashion_mnist_f1:.6f}")